In [1]:
import pandas as pd
import secrets
import string

In [2]:
def pseudonymise_dataset(df):

    df = df.copy()

    # Generate unique random numeric IDs
    def generate_numeric_ids(values):
        unique_values = pd.Series(values).dropna().unique()
        mapping = {}
        used = set()

        for value in unique_values:
            length = len(str(value))

            while True:
                new_id = ''.join(
                    secrets.choice(string.digits)
                    for _ in range(length)
                )

                if new_id not in used:
                    mapping[value] = new_id
                    used.add(new_id)
                    break

        return mapping

    # Generate unique random alphanumeric aliases
    def generate_aliases(values, prefix):
        unique_values = pd.Series(values).dropna().unique()
        mapping = {}
        used = set()

        for value in unique_values:
            while True:
                alias = prefix + ''.join(
                    secrets.choice(
                        string.ascii_uppercase + string.digits
                    )
                    for _ in range(10)
                )

                if alias not in used:
                    mapping[value] = alias
                    used.add(alias)
                    break

        return mapping

    # Preserve original values for consistent mapping
    original_student_ids = df["Student ID"].copy()
    original_opportunity_ids = df["Opportunity ID"].copy()
    original_deal_owners = df["Deal Owner"].copy()

    # Pseudonymise Student IDs
    student_mapping = generate_numeric_ids(original_student_ids)

    df["Student ID"] = original_student_ids.map(student_mapping)

    # Pseudonymise Opportunity IDs
    opportunity_mapping = generate_numeric_ids(
        original_opportunity_ids
    )

    df["Opportunity ID"] = original_opportunity_ids.map(
        opportunity_mapping
    )

    # Pseudonymise Student Names
    # Each unique Student ID gets one consistent synthetic name
    student_name_mapping = {}

    for student_id in original_student_ids.dropna().unique():
        student_name_mapping[student_id] = (
            "STUDENT_" + ''.join(
                secrets.choice(string.ascii_uppercase)
                for _ in range(8)
            )
        )

    df["Student Name"] = original_student_ids.map(
        student_name_mapping
    )

    # Pseudonymise Deal Owners (These are the agency's agents who are assigned to the applications.)
    deal_owner_mapping = generate_aliases(
        original_deal_owners,
        "OWNER_"
    )

    df["Deal Owner"] = original_deal_owners.map(
        deal_owner_mapping
    )

    return df

In [3]:
df = pd.read_csv(
    "/Users/nkirote/Desktop/_EDUCATION/MSc DSA/MODULE 4/Applied Machine Learning in Data Science/Student_Applications_Success_Predictor_11_2026/data/Raw_Deals_Data.csv",
    dtype={
        "Student ID": "string",
        "Opportunity ID": "string"
    }
)

In [4]:
df_pseudonymised = pseudonymise_dataset(df)

In [5]:
df_pseudonymised.to_csv(
    "/Users/nkirote/Desktop/_EDUCATION/MSc DSA/MODULE 4/Applied Machine Learning in Data Science/Student_Applications_Success_Predictor_11_2026/data/higher_studies_inc_pseudonymised_data.csv",
    index=False
)

print("Pseudonymisation completed successfully!")
print(f"Original deals data: {len(df)}")
print(f"Pseudonymised data: {len(df_pseudonymised)}")
print("Output saved as data/higher_studies_inc_pseudonymised_data.csv")


Pseudonymisation completed successfully!
Original deals data: 6024
Pseudonymised data: 6024
Output saved as data/higher_studies_inc_pseudonymised_data.csv
